In [1]:
try:
    from databricks.sdk import WorkspaceClient
except ImportError:
    !pip install databricks-sdk
    from databricks.sdk import WorkspaceClient

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 594.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.0/822.0 kB 10.8 MB/s eta 0:00:00


In [4]:
# Importing Libraries
import json
import requests
import pandas as pd
from databricks.sdk import WorkspaceClient
from google.colab import userdata

# Defining functions

In [23]:
# ----- # ----- # ----- Get Jobs for United Health Group ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_uhg(limit=100, offset=0):
    response = ''
    try:
        url = 'https://jobsapi-internal.m-cloud.io/api/job'
        params = {
            # 'callback': 'CWS.jobs.jobCallback',
            'facet[]': 'ats_portalid:Smashfly',
            'sortfield': 'open_date',
            'sortorder': 'descending',
            'Limit': limit,
            'Organization': '2071',
            'offset': offset,
            'useBooleanKeywordSearch': True
        }

        headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': '*/*',
        'Referer': 'https://careers.unitedhealthgroup.com/job-search-results/',
        'Accept-Encoding': 'gzip, deflate, br, zstd'
        }

        response = requests.get(url, params=params, headers=headers)
        response_json = json.loads(response.text)
        if 'queryResult' in response_json:
          if len(response_json['queryResult']) > 0:
            return True, response_json
          else:
            return False, response_json
        return False, {'Error': response.text}
    except Exception as e:
        print(str(e))
        print(f"Unable to connect to United Health Group: {response.text}")
        return False, {'Error': response.text}

# ----- # ----- # ----- Fetch Job List ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_jobs():
    limit = 100
    offset = 0
    job_df_list = []

    while True:
        if offset%100 == 0:
          print(f'Offset: {offset}, Count: {limit + offset}')
        job_bool, job_json = jobs_uhg(limit, offset)
        if job_bool:
          job_df_list.extend(job_json['queryResult'])
          offset += limit
        else:
          break

    # Concatenate the dataset and clean
    jobs = pd.DataFrame(job_df_list)
    jobs.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/uhg_jobs.parquet', index=False)
    print(f'Total Jobs: {len(jobs)}')
    return jobs

In [25]:
# ----- # ----- # ----- Upload file to Databricks ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def upload_data(data_file):
  w = WorkspaceClient(
      host = userdata.get('DATABRICKS_HOST'),
      token = userdata.get('DATABRICKS_TOKEN')
  )

  volume_path = "/Volumes/workspace/jobs_raw/job_files/" + data_file.split('/')[-1]
  with open(data_file, "rb") as f:
      w.files.upload(volume_path, f)

  print(f"Successfully uploaded {data_file} to {volume_path}")

# New Architecture

In [26]:
job_frame = fetch_jobs()
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/uhg_jobs.parquet')

Offset: 0, Count: 100
Offset: 100, Count: 200
Offset: 200, Count: 300
Offset: 300, Count: 400
Offset: 400, Count: 500
Offset: 500, Count: 600
Offset: 600, Count: 700
Offset: 700, Count: 800
Offset: 800, Count: 900
Offset: 900, Count: 1000
Offset: 1000, Count: 1100
Offset: 1100, Count: 1200
Offset: 1200, Count: 1300
Offset: 1300, Count: 1400
Offset: 1400, Count: 1500
Offset: 1500, Count: 1600
Offset: 1600, Count: 1700
Offset: 1700, Count: 1800
Offset: 1800, Count: 1900
Offset: 1900, Count: 2000
Offset: 2000, Count: 2100
Offset: 2100, Count: 2200
Offset: 2200, Count: 2300
Offset: 2300, Count: 2400
Offset: 2400, Count: 2500
Offset: 2500, Count: 2600
Offset: 2600, Count: 2700
Offset: 2700, Count: 2800
Offset: 2800, Count: 2900
Offset: 2900, Count: 3000
Offset: 3000, Count: 3100
Offset: 3100, Count: 3200
Offset: 3200, Count: 3300
Offset: 3300, Count: 3400
Offset: 3400, Count: 3500
Offset: 3500, Count: 3600
Offset: 3600, Count: 3700
Offset: 3700, Count: 3800
Offset: 3800, Count: 3900
Offset: